# CGR: Constraint-Aware Generative Re-ranking

**Paper:** *Constraint-Aware Generative Re-ranking for Multi-Objective Optimization in Advertising Feeds* (Bilibili, arXiv:2603.04227)

This notebook walks through the key components of the CGR implementation:

1. **Domain types** — Items, constraints, candidate sets
2. **Model architecture** — Hierarchical attention + PLE fusion + reward heads
3. **Training** — Multi-task BCE loss on logged impression data
4. **Inference** — Two-stage bounded decoding with constraint-aware pruning
5. **End-to-end example** — Train on synthetic data, run inference, verify constraints

In [1]:
import subprocess, sys, os

# Install dependencies from requirements.txt
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

# Ensure the project root is on the path so we can import cgr as a package
sys.path.insert(0, os.path.abspath(".."))

import torch, numpy
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device:    {device}")

PyTorch version: 2.11.0+cpu
CUDA available:  False
Using device:    cpu


## 1. Domain Types — Items, Constraints, Candidate Sets

CGR operates on **Items** that carry pre-computed embeddings from upstream models (DLRM item tower, user tower, context encoder) plus scalar signals from the ad auction and business policy.

Key classes:
- `Item` — a single feed candidate (organic or ad) with upstream embeddings + auction scalars
- `AdConstraints` — business rules: max ads per page (K), minimum spacing (Δ), position bounds, etc.
- `CandidateSet` — the organic list + ad candidates for a single user request

In [3]:
from cgr.data_types import Item, ItemType, AdConstraints, CandidateSet

# Embedding dimensions (would match your upstream tower output sizes)
ITEM_EMB_DIM = 16
USER_EMB_DIM = 8
CTX_EMB_DIM = 4

# --- Create an organic item ---
organic_item = Item(
    item_id=0,
    item_type=ItemType.ORGANIC,
    item_emb=torch.randn(ITEM_EMB_DIM),    # v_i from item tower
    user_emb=torch.randn(USER_EMB_DIM),     # u_i from user tower
    context_emb=torch.randn(CTX_EMB_DIM),   # c_i from context encoder
    bid=0.0,                                 # organic items have no bid
    engagement_score=1.2,                    # predicted engagement from upstream
    ad_penalty_weight=0.0,                   # no penalty for organic
)
print(f"Organic item: id={organic_item.item_id}, is_ad={organic_item.is_ad}")
print(f"  item_emb shape: {organic_item.item_emb.shape}")
print(f"  user_emb shape: {organic_item.user_emb.shape}")
print(f"  context_emb shape: {organic_item.context_emb.shape}")

# --- Create an ad item ---
ad_item = Item(
    item_id=100,
    item_type=ItemType.AD,
    item_emb=torch.randn(ITEM_EMB_DIM),
    user_emb=torch.randn(USER_EMB_DIM),
    context_emb=torch.randn(CTX_EMB_DIM),
    bid=3.0,                                 # s_i: advertiser bid price
    engagement_score=0.5,                    # n_i: expected engagement
    ad_penalty_weight=0.3,                   # d_i: business policy penalty
)
print(f"\nAd item: id={ad_item.item_id}, is_ad={ad_item.is_ad}, bid={ad_item.bid}")

Organic item: id=0, is_ad=False
  item_emb shape: torch.Size([16])
  user_emb shape: torch.Size([8])
  context_emb shape: torch.Size([4])

Ad item: id=100, is_ad=True, bid=3.0


In [7]:
# --- Define business constraints (Section 3.4) ---
constraints = AdConstraints(
    max_ads_per_list=2,    # K: at most 2 ads per page
    min_ad_spacing=3,      # Δ: ads must be ≥3 positions apart
    min_ad_position=1,     # ads cannot appear at position 0 (top of feed)
    max_ad_position=8,     # ads cannot appear past position 8
    max_large_ads=1,       # at most 1 large-format ad
    ad_density_limit=0.3,  # at most 30% of the page can be ads
)

print("Feasible ad positions for a 10-item list:", constraints.get_feasible_ad_positions(10))
print()

# Check spacing: positions [2, 5] with min_spacing=3 → OK (5-2=3 ≥ 3)
print("Spacing check [2, 5]:", constraints.check_spacing([2, 5]))
# Check spacing: positions [2, 4] with min_spacing=3 → FAIL (4-2=2 < 3)
print("Spacing check [2, 4]:", constraints.check_spacing([2, 4]))
# Load check: 2 ads with K=2 → OK
print("Load check (2 ads):", constraints.check_load(2))
# Load check: 3 ads with K=2 → FAIL
print("Load check (3 ads):", constraints.check_load(3))

Feasible ad positions for a 10-item list: [1, 2, 3, 4, 5, 6, 7, 8]

Spacing check [2, 5]: True
Spacing check [2, 4]: False
Load check (2 ads): True
Load check (3 ads): False


## 2. Model Architecture

The CGR model (Section 4) is a unified listwise autoregressive network:

```
Upstream embeddings [v_i, u_i, c_i]
        ↓
  Adapter projections → d_model    (thin linear layers, NOT full encoders)
  + positional embedding p_i       (the only embedding learned within CGR)
        ↓
  Hierarchical Attention blocks     (shared SA + task-specific SA + cross-attn + LSA)
        ↓
  PLE Fusion                        (gated expert mixing → h_exp, h_clk)
        ↓
  Reward Heads                      (MLP → p_exp, p_clk)
        ↓
  R(A) = Σ(V_i + N_i − P_i)        (list-level reward, no separate evaluator)
```

In [8]:
from cgr.model import CGRModel

model = CGRModel(
    item_emb_dim=ITEM_EMB_DIM,    # must match upstream item tower output dim
    user_emb_dim=USER_EMB_DIM,    # must match upstream user tower output dim
    context_emb_dim=CTX_EMB_DIM,  # must match upstream context encoder output dim
    d_model=32,                    # internal hidden dimension
    n_heads=4,                     # attention heads per MHA layer
    n_layers=2,                    # number of HierarchicalAttentionBlock layers
    max_list_len=11,               # maximum sequence length L (paper uses 11)
    dropout=0.1,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel structure:\n{model}")

Total parameters:     62,733
Trainable parameters: 62,733

Model structure:
CGRModel(
  (item_adapter): Linear(in_features=16, out_features=32, bias=True)
  (user_adapter): Linear(in_features=8, out_features=32, bias=True)
  (context_adapter): Linear(in_features=4, out_features=32, bias=True)
  (position_emb): Embedding(11, 32)
  (item_type_emb): Embedding(4, 32)
  (input_fusion): Sequential(
    (0): Linear(in_features=128, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (attn_blocks): ModuleList(
    (0-1): 2 x HierarchicalAttentionBlock(
      (shared_self_attn): MultiHeadSelfAttention(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
      )
      (exp_self_attn): MultiHeadSelfAttention(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
      )
      (clk_

In [9]:
# --- Forward pass demo ---
# Simulate a batch of 2 sequences, each with 8 items
B, L = 2, 8
item_embs = torch.randn(B, L, ITEM_EMB_DIM)
user_embs = torch.randn(B, L, USER_EMB_DIM)
context_embs = torch.randn(B, L, CTX_EMB_DIM)
item_types = torch.zeros(B, L, dtype=torch.long)
item_types[:, 3] = 1  # ad at position 3

model.eval()
with torch.no_grad():
    p_exp, p_clk = model(item_embs, user_embs, context_embs, item_types)

print(f"Input shapes:  item_embs={item_embs.shape}, user_embs={user_embs.shape}, context_embs={context_embs.shape}")
print(f"Output shapes: p_exp={p_exp.shape}, p_clk={p_clk.shape}")
print(f"\nSample p_exp (sequence 0): {p_exp[0].numpy().round(3)}")
print(f"Sample p_clk (sequence 0): {p_clk[0].numpy().round(3)}")

Input shapes:  item_embs=torch.Size([2, 8, 16]), user_embs=torch.Size([2, 8, 8]), context_embs=torch.Size([2, 8, 4])
Output shapes: p_exp=torch.Size([2, 8]), p_clk=torch.Size([2, 8])

Sample p_exp (sequence 0): [0.58  0.553 0.637 0.637 0.531 0.549 0.507 0.644]
Sample p_clk (sequence 0): [0.585 0.619 0.516 0.547 0.47  0.597 0.596 0.549]


## 3. Computing List-Level Reward

The reward R(A) combines three per-position components (Eqs. 17-20):

| Component | Formula | Meaning |
|-----------|---------|---------|
| V_i (monetization) | `p_clk * p_exp * bid` (CPA) or `p_exp * bid` (CPM) | Ad revenue |
| N_i (engagement) | `p_exp * p_clk * engagement_score` | User satisfaction |
| P_i (penalty) | `ad_penalty_weight * p_exp` | Cost of showing an ad |

**R(A) = Σ(V_i + N_i − P_i)** — the model maximises revenue + engagement while penalising ad overexposure.

In [ ]:
from cgr.model import ExposureClickHead

# Use the p_exp, p_clk from the forward pass above
bids = torch.zeros(B, L)
bids[:, 3] = 3.0  # only the ad at position 3 has a bid

engagement_scores = torch.ones(B, L) * 1.0
ad_penalty_weights = torch.zeros(B, L)
ad_penalty_weights[:, 3] = 0.3  # penalty only for the ad

is_cpa = torch.zeros(B, L)
is_cpa[:, 3] = 1.0  # the ad uses CPA billing

reward = ExposureClickHead.compute_reward(p_exp, p_clk, bids, engagement_scores, ad_penalty_weights, is_cpa)
print(f"List-level rewards: {reward.numpy().round(4)}")
print(f"  Sequence 0: R(A) = {reward[0].item():.4f}")
print(f"  Sequence 1: R(A) = {reward[1].item():.4f}")

List-level rewards: [3.4476 3.4029]
  Sequence 0: R(A) = 3.4476
  Sequence 1: R(A) = 3.4029


## 4. Training on Logged Impression Data

In production, training data consists of logged impression sequences:
- Pre-computed embeddings (from upstream DLRM pipeline)
- Binary exposure/click labels (from user interaction logs)

The loss is a weighted multi-task BCE:  `L = λ_exp · BCE(p_exp, y_exp) + λ_clk · BCE(p_clk, y_clk)`

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from cgr.train import CGRTrainer, TrainConfig

# --- Create synthetic training data ---
# In production these would be logged embeddings + labels from your serving pipeline
n_samples, list_len = 256, 8

train_data = TensorDataset(
    torch.randn(n_samples, list_len, ITEM_EMB_DIM),   # item embeddings
    torch.randn(n_samples, list_len, USER_EMB_DIM),    # user embeddings
    torch.randn(n_samples, list_len, CTX_EMB_DIM),     # context embeddings
    torch.zeros(n_samples, list_len, dtype=torch.long), # item types (all organic for simplicity)
    (torch.rand(n_samples, list_len) > 0.3).float(),   # exposure labels (~70% exposed)
    (torch.rand(n_samples, list_len) > 0.7).float(),   # click labels (~30% clicked)
)

def collate_fn(batch):
    item_e, user_e, ctx_e, types, exp, clk = zip(*batch)
    return {
        "item_embs": torch.stack(item_e),
        "user_embs": torch.stack(user_e),
        "context_embs": torch.stack(ctx_e),
        "item_types": torch.stack(types),
        "exp_labels": torch.stack(exp),
        "clk_labels": torch.stack(clk),
    }

loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_fn)
print(f"Training samples: {n_samples}, batches per epoch: {len(loader)}")

In [ ]:
# --- Train the model ---
config = TrainConfig(
    lr=1e-3,
    weight_decay=1e-5,
    lambda_exp=1.0,   # equal weight on exposure and click losses
    lambda_clk=1.0,
    epochs=10,
    batch_size=32,
    grad_clip=1.0,
)

# Re-initialise model for a clean training run
model = CGRModel(
    item_emb_dim=ITEM_EMB_DIM,
    user_emb_dim=USER_EMB_DIM,
    context_emb_dim=CTX_EMB_DIM,
    d_model=32,
    n_heads=4,
    n_layers=2,
    max_list_len=11,
)
trainer = CGRTrainer(model, config, device)

print("Epoch  | Loss    | L_exp   | L_clk")
print("-------|---------|---------|--------")
for epoch in range(config.epochs):
    metrics = trainer.train_epoch(loader)
    print(f"  {epoch+1:2d}   | {metrics['loss']:.4f}  | {metrics['loss_exp']:.4f}  | {metrics['loss_clk']:.4f}")

## 5. Two-Stage Inference

The inference pipeline (Figure 2) has two stages:

**Stage I — Constrained Single-Ad Insertion** (Section 7.1):
- Takes the pre-ranked organic list from upstream
- Tries inserting each regular ad at every feasible position
- Evaluates reward for each candidate via batched forward pass
- Selects the best single-ad (or no-ad) sequence

**Stage II — Bounded Generative Decoding** (Section 7.2):
- Expands to 4 list families: large-ad, double-ad, single-ad, no-ad
- Applies hard constraint filtering (Section 6.1) before evaluation
- Applies reward upper-bound pruning (Section 6.2) to skip dominated candidates
- Returns the globally optimal feasible sequence

Because K ≤ 2, complexity is O(K·L) instead of O(N!) — feasible under <40ms P99 latency.

In [ ]:
from cgr.inference import cgr_inference, stage1_constrained_insertion, stage2_bounded_decoding

# --- Build a realistic candidate set ---
def make_item(item_id, item_type, bid=0.0, engagement=1.0, penalty=0.0):
    return Item(
        item_id=item_id,
        item_type=item_type,
        item_emb=torch.randn(ITEM_EMB_DIM),
        user_emb=torch.randn(USER_EMB_DIM),
        context_emb=torch.randn(CTX_EMB_DIM),
        bid=bid,
        engagement_score=engagement,
        ad_penalty_weight=penalty,
    )

# 8 organic items (pre-ranked by upstream DLRM)
organic_items = [make_item(i, ItemType.ORGANIC, engagement=1.0 + 0.1 * i) for i in range(8)]

# 4 ad candidates from the auction system
ad_candidates = [
    make_item(100, ItemType.AD,       bid=2.5, engagement=0.5, penalty=0.3),
    make_item(101, ItemType.AD,       bid=3.0, engagement=0.3, penalty=0.4),
    make_item(102, ItemType.LARGE_AD, bid=5.0, engagement=0.8, penalty=0.5),
    make_item(103, ItemType.ECOM_AD,  bid=1.5, engagement=0.6, penalty=0.2),
]

candidates = CandidateSet(organic_items=organic_items, ad_candidates=ad_candidates)
print(f"Candidate set: {candidates.num_organic} organic + {candidates.num_ads} ads")

In [ ]:
# --- Stage I: Constrained single-ad insertion ---
print("=" * 60)
print("Stage I: Constrained Single-Ad Insertion")
print("=" * 60)

intermediate = stage1_constrained_insertion(
    model, organic_items, ad_candidates, constraints, device
)

print(f"Best single-ad reward: {intermediate.reward:.4f}")
print(f"Ad positions: {intermediate.ad_positions}")
print(f"Sequence ({len(intermediate.items)} items):")
for i, item in enumerate(intermediate.items):
    marker = " ← AD" if item.is_ad else ""
    print(f"  [{i}] item_{item.item_id} ({item.item_type.name}){marker}")

In [ ]:
# --- Stage II: Bounded generative decoding ---
print("=" * 60)
print("Stage II: Bounded Generative Decoding")
print("=" * 60)

final = stage2_bounded_decoding(
    model, intermediate, ad_candidates, constraints, device, use_pruning=True
)

print(f"Final reward: {final.reward:.4f}")
print(f"Ad positions: {final.ad_positions}")
print(f"Sequence ({len(final.items)} items):")
for i, item in enumerate(final.items):
    marker = f" ← {item.item_type.name} (bid={item.bid})" if item.is_ad else ""
    print(f"  [{i}] item_{item.item_id}{marker}")

print(f"\nConstraints satisfied: {constraints.is_feasible(final.items)}")

## 6. Full Pipeline (One Call)

The `cgr_inference()` function wraps both stages into a single call:

In [ ]:
# --- Full inference in one call ---
result = cgr_inference(
    model=model,
    candidates=candidates,
    constraints=constraints,
    device=device,
    use_pruning=True,  # set to False to disable reward upper-bound pruning
)

print(f"Optimal reward: {result.reward:.4f}")
print(f"Number of ads:  {result.num_ads}")
print(f"Ad positions:   {result.ad_positions}")
print(f"Constraints OK: {constraints.is_feasible(result.items)}")
print(f"\nFinal feed:")
for i, item in enumerate(result.items):
    if item.is_ad:
        print(f"  [{i}] >> {item.item_type.name} (id={item.item_id}, bid={item.bid}, penalty={item.ad_penalty_weight})")
    else:
        print(f"  [{i}]    ORGANIC (id={item.item_id}, engagement={item.engagement_score})")

## 7. Comparing Pruning vs. No Pruning

Reward pruning (Section 6.2) skips candidates whose upper-bound reward can't beat the current best. Let's compare inference with and without it:

In [ ]:
import time

# With pruning
t0 = time.perf_counter()
result_pruned = cgr_inference(model, candidates, constraints, device, use_pruning=True)
t_pruned = time.perf_counter() - t0

# Without pruning
t0 = time.perf_counter()
result_no_prune = cgr_inference(model, candidates, constraints, device, use_pruning=False)
t_no_prune = time.perf_counter() - t0

print(f"With pruning:    reward={result_pruned.reward:.4f}  time={t_pruned*1000:.1f}ms")
print(f"Without pruning: reward={result_no_prune.reward:.4f}  time={t_no_prune*1000:.1f}ms")
print(f"\nRewards match: {abs(result_pruned.reward - result_no_prune.reward) < 1e-4}")
print("(Pruning preserves optimality — Theorem 9.1)")

## 8. Experimenting with Different Constraints

See how the optimal sequence changes under different business rules:

In [ ]:
# --- Strict: only 1 ad allowed ---
strict = AdConstraints(max_ads_per_list=1, min_ad_spacing=3, min_ad_position=1, max_ad_position=8)
result_strict = cgr_inference(model, candidates, strict, device)
print(f"Strict (K=1):  {result_strict.num_ads} ad(s), reward={result_strict.reward:.4f}, positions={result_strict.ad_positions}")

# --- Relaxed: up to 2 ads, smaller spacing ---
relaxed = AdConstraints(max_ads_per_list=2, min_ad_spacing=2, min_ad_position=1, max_ad_position=8)
result_relaxed = cgr_inference(model, candidates, relaxed, device)
print(f"Relaxed (Δ=2): {result_relaxed.num_ads} ad(s), reward={result_relaxed.reward:.4f}, positions={result_relaxed.ad_positions}")

# --- No ads allowed ---
no_ads = AdConstraints(max_ads_per_list=0)
result_no_ads = cgr_inference(model, candidates, no_ads, device)
print(f"No ads (K=0):  {result_no_ads.num_ads} ad(s), reward={result_no_ads.reward:.4f}")

print(f"\nAll constraints satisfied:")
print(f"  Strict:  {strict.is_feasible(result_strict.items)}")
print(f"  Relaxed: {relaxed.is_feasible(result_relaxed.items)}")
print(f"  No ads:  {no_ads.is_feasible(result_no_ads.items)}")

## Summary

| Module | Paper Section | What it does |
|--------|:------------:|--------------|
| `cgr.types` | 3 | `Item`, `AdConstraints`, `CandidateSet` — domain types + constraint validation |
| `cgr.model` | 4 | `CGRModel` — hierarchical attention + PLE fusion + unified reward heads |
| `cgr.train` | 4 | `CGRTrainer` — multi-task BCE training loop |
| `cgr.inference` | 5, 6, 7 | Two-stage bounded decoding + constraint-aware reward pruning |

**Key design choices:**
- CGR receives **pre-computed embeddings** from upstream towers — it does not learn item/user/context representations from scratch
- The model **unifies generation and evaluation** in a single forward pass (no separate evaluator network)
- **Bounded decoding** exploits K ≤ 2 to reduce O(N!) → O(K·L) complexity
- **Constraint-aware pruning** maintains optimality (Theorem 9.1) while reducing latency